In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import warnings

In [2]:
df = pd.read_csv('data/StudentsPerformance.csv')

In [3]:
df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [7]:
X = df.drop(columns=['math score'])

In [8]:
X.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,74
1,female,group C,some college,standard,completed,90,88
2,female,group B,master's degree,standard,none,95,93
3,male,group A,associate's degree,free/reduced,none,57,44
4,male,group C,some college,standard,none,78,75


In [19]:
y = df['math score']

In [10]:
num_features = X.select_dtypes(exclude='object').columns
cat_features = X.select_dtypes(include='object').columns

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", categorical_transformer, cat_features),
        ("StandardScaler", numeric_transformer, num_features)
    ]
)

C:\Users\ahsan\AppData\Local\Temp\ipykernel_29224\3059956823.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_features = X.select_dtypes(include='object').columns


In [13]:
X = preprocessor.fit_transform(X)

In [16]:
X.shape

(1000, 19)

In [17]:
column_names = preprocessor.get_feature_names_out()
print(column_names)
print(f"Total count: {len(column_names)}")

['OneHotEncoder__gender_female' 'OneHotEncoder__gender_male'
 'OneHotEncoder__race/ethnicity_group A'
 'OneHotEncoder__race/ethnicity_group B'
 'OneHotEncoder__race/ethnicity_group C'
 'OneHotEncoder__race/ethnicity_group D'
 'OneHotEncoder__race/ethnicity_group E'
 "OneHotEncoder__parental level of education_associate's degree"
 "OneHotEncoder__parental level of education_bachelor's degree"
 'OneHotEncoder__parental level of education_high school'
 "OneHotEncoder__parental level of education_master's degree"
 'OneHotEncoder__parental level of education_some college'
 'OneHotEncoder__parental level of education_some high school'
 'OneHotEncoder__lunch_free/reduced' 'OneHotEncoder__lunch_standard'
 'OneHotEncoder__test preparation course_completed'
 'OneHotEncoder__test preparation course_none'
 'StandardScaler__reading score' 'StandardScaler__writing score']
Total count: 19


In [22]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((800, 19), (200, 19), (800,), (200,))

In [21]:
def evaluate_model(true, predicted):
    r2 = r2_score(true, predicted)
    mse = mean_squared_error(true, predicted)
    mae = mean_absolute_error(true, predicted)
    rmse = np.sqrt(mse)
    return mae, rmse, r2

In [23]:
models = {
    "Linear Regression": LinearRegression(),
    "KNN Regressor": KNeighborsRegressor(),
    "Decision Tree Regressor": DecisionTreeRegressor(random_state=42),
    "Random Forest Regressor": RandomForestRegressor(random_state=42),
    "AdaBoost Regressor": AdaBoostRegressor(random_state=42),
    "SVR": SVR(),
    "CatBoost Regressor": CatBoostRegressor(random_state=42, verbose=0),
    "XGBoost Regressor": XGBRegressor(random_state=42, verbosity=0)
}

model_list = []
r2_list = []

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    model_train_mae, model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)
    model_test_mae, model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)

    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])

    print("Model performance on training set")
    print(f"MAE: {model_train_mae}")
    print(f"RMSE: {model_train_rmse}")
    print(f"R2 Score: {model_train_r2}")
    print("\n")
    print("-----------------------------------------------------")

    print("Model performance on testing set")
    print(f"MAE: {model_test_mae}")
    print(f"RMSE: {model_test_rmse}")
    print(f"R2 Score: {model_test_r2}")
    r2_list.append(model_test_r2)
    print("-----------------------------------------------------")

    print('='*35)
    print('\n')

Linear Regression
Model performance on training set
MAE: 4.266711846071957
RMSE: 5.323050852720514
R2 Score: 0.8743172040139593


-----------------------------------------------------
Model performance on testing set
MAE: 4.21476314247485
RMSE: 5.393993869732843
R2 Score: 0.8804332983749565
-----------------------------------------------------


KNN Regressor
Model performance on training set
MAE: 4.516749999999999
RMSE: 5.707683417990174
R2 Score: 0.8554978341651085


-----------------------------------------------------
Model performance on testing set
MAE: 5.621
RMSE: 7.253040741647602
R2 Score: 0.7838129945787431
-----------------------------------------------------


Decision Tree Regressor
Model performance on training set
MAE: 0.01875
RMSE: 0.2795084971874737
R2 Score: 0.9996534669718089


-----------------------------------------------------
Model performance on testing set
MAE: 6.195
RMSE: 7.714596554584044
R2 Score: 0.7554229007834358
-----------------------------------------

In [24]:
pd.DataFrame(list(zip(model_list, r2_list)), columns=['Model', 'R2 Score']).sort_values(by='R2 Score', ascending=False)

,Model,R2 Score
0,Linear Regression,0.880433
3,Random Forest Regressor,0.851185
6,CatBoost Regressor,0.850185
4,AdaBoost Regressor,0.845283
7,XGBoost Regressor,0.827797
1,KNN Regressor,0.783813
2,Decision Tree Regressor,0.755423
5,SVR,0.728600
